# 05 — From Hidden State to Probabilities

**Description:** Transform a model's hidden state into vocabulary logits, implement numerically stable softmax, and inspect the resulting next-token probabilities.
**Level:** Beginner
**Tags:** Language Models, Unembedding, Logits, Softmax, Probabilities

A transformer produces one **hidden-state vector** at each token position. For next-token prediction, the last hidden state must become one probability per vocabulary token:

$$\text{hidden state} \rightarrow \text{logits} \rightarrow \text{probabilities}$$

This notebook makes both transformations explicit. By the end, you will be able to:

- use an unembedding matrix to produce vocabulary-sized logits;
- explain logits as unnormalized token scores;
- implement softmax safely with NumPy;
- inspect, rank, and visualize next-token probabilities;
- track shapes for sequences and batches; and
- connect the output distribution to cross-entropy training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=4, suppress=True)
torch.set_printoptions(precision=4, sci_mode=False)
plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(7)
torch.manual_seed(7)

## 1. The model's last hidden state

After processing a context such as `the robot`, a transformer holds a vector for its final position. This **hidden state** summarizes information the network considers useful for predicting what follows.

Real models use hundreds or thousands of hidden features. We will use four so every calculation remains visible. The individual coordinates do not need simple human-readable meanings.

In [ ]:
context = "the robot"
hidden_state = np.array([0.8, -0.4, 1.2, 0.5])
d_model = hidden_state.shape[0]

print("context:     ", context)
print("hidden state:", hidden_state)
print("shape:       ", hidden_state.shape)
print("d_model:     ", d_model)

## 2. The output vocabulary

The model must assign a score to every possible next token. Our vocabulary is tiny enough to inspect directly. A real language model may have tens or hundreds of thousands of vocabulary entries, but the shape rule is identical.

In [ ]:
vocabulary = ["learns", "writes", "runs", "quickly", ".", "<EOS>"]
token_to_id = {token: i for i, token in enumerate(vocabulary)}
vocab_size = len(vocabulary)

print("vocabulary size:", vocab_size)
print(token_to_id)

## 3. Unembedding maps features to token scores

The embedding matrix in Notebook 03 mapped token IDs **into** vectors. An **unembedding** or output projection maps a hidden vector **out to** one score per vocabulary token.

We will store one token-scoring vector in each column of matrix $W_U$:

$$\text{logits} = hW_U + b_U$$

Shapes: `(d_model,) @ (d_model, vocab_size) + (vocab_size,) → (vocab_size,)`.

In [ ]:
unembedding = np.array([
    [ 1.0,  0.4, -0.6,  0.1, -0.2,  0.0],
    [-0.3,  0.8,  0.2, -0.5,  0.1,  0.4],
    [ 0.7, -0.2,  0.3,  0.6, -0.4, -0.1],
    [ 0.2,  0.1,  0.5,  0.3,  0.7,  0.6],
])
output_bias = np.array([0.1, -0.1, 0.0, 0.2, -0.2, 0.0])

print("hidden shape:     ", hidden_state.shape)
print("unembedding shape:", unembedding.shape)
print("bias shape:       ", output_bias.shape)

### Predict the output shape

Before running the next cell, use the matrix-multiplication rule to predict the shape. Why must the result contain exactly one value per vocabulary token?

In [ ]:
logits = hidden_state @ unembedding + output_bias

print("logits shape:", logits.shape)
for token, logit in zip(vocabulary, logits):
    print(f"{token:8s} → {logit: .3f}")

## 4. Each logit is a dot product

Every vocabulary token has one column in the unembedding matrix. Its logit is the dot product between that column and the hidden state, plus the token's bias.

A high logit means the hidden state aligns well with that token's output vector. Logits are scores on an unrestricted scale: they can be negative, positive, and need not sum to anything.

In [ ]:
token = "learns"
token_id = token_to_id[token]
token_output_vector = unembedding[:, token_id]
manual_logit = hidden_state @ token_output_vector + output_bias[token_id]

print("hidden state:       ", hidden_state)
print("token output vector:", token_output_vector)
print("manual logit:       ", manual_logit)
print("matrix result:      ", logits[token_id])
print("match:              ", np.allclose(manual_logit, logits[token_id]))

### Your turn: inspect another token

Change `token` and inspect the elementwise contributions to its logit. Which hidden feature contributes most positively? Which contributes most negatively?

In [ ]:
token = "writes"  # Edit me
token_id = token_to_id[token]
contributions = hidden_state * unembedding[:, token_id]

print("feature contributions:", contributions)
print("contribution sum:    ", contributions.sum())
print("bias:                ", output_bias[token_id])
print("final logit:         ", contributions.sum() + output_bias[token_id])

## 5. Softmax turns scores into probabilities

Softmax exponentiates each logit and divides by the sum of all exponentials:

$$P(i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Exponentiation makes every value positive. Division makes the results sum to 1. Larger logits receive larger probabilities, and differences between logits—not their absolute level—determine the distribution.

In [ ]:
def softmax_first_attempt(values):
    exponentials = np.exp(values)
    return exponentials / exponentials.sum()

probabilities = softmax_first_attempt(logits)

for token, logit, probability in zip(vocabulary, logits, probabilities):
    print(f"{token:8s} | logit={logit:6.3f} | probability={probability:6.2%}")
print("sum:", probabilities.sum())

## 6. Why exponentiation changes competition

The exponential function converts additive logit gaps into multiplicative odds. If token A's logit is 2 greater than token B's, its unnormalized weight is $e^2 \approx 7.39$ times larger.

For any two tokens $a$ and $b$:

$$\frac{P(a)}{P(b)} = e^{z_a-z_b}$$

In [ ]:
token_a, token_b = "learns", "writes"
a, b = token_to_id[token_a], token_to_id[token_b]
logit_gap = logits[a] - logits[b]

print("logit gap:                ", logit_gap)
print("exp(logit gap):           ", np.exp(logit_gap))
print("probability ratio P(a)/P(b):", probabilities[a] / probabilities[b])

## 7. Implement numerically stable softmax

The first implementation can overflow when a logit is large because `exp(1000)` cannot fit in an ordinary floating-point value. Softmax is unchanged if the same constant is subtracted from every logit. We therefore subtract the maximum first:

$$\operatorname{softmax}(z) = \operatorname{softmax}(z - \max(z))$$

The largest shifted logit becomes zero, so its exponential is 1 and no exponential is greater than 1.

In [ ]:
def softmax(values, axis=-1):
    values = np.asarray(values, dtype=float)
    shifted = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)

stable_probabilities = softmax(logits)
print("stable result: ", stable_probabilities)
print("matches simple:", np.allclose(stable_probabilities, probabilities))
print("sums to one:  ", stable_probabilities.sum())

### See numerical stability matter

The warnings from the naive calculation are part of the demonstration. The stable version produces finite probabilities while preserving the relative preference encoded by the logits.

In [ ]:
large_logits = np.array([1000.0, 1001.0, 1002.0])

with np.errstate(over="ignore", invalid="ignore"):
    naive_result = softmax_first_attempt(large_logits)
stable_result = softmax(large_logits)

print("naive: ", naive_result)
print("stable:", stable_result)
print("finite:", np.isfinite(stable_result).all())

## 8. Softmax ignores a shared offset

Adding the same number to every logit does not change the probabilities. Only relative gaps matter. This explains why subtracting the maximum is mathematically safe.

In [ ]:
offsets = [-100.0, 0.0, 37.5, 1000.0]
reference = softmax(logits)

for offset in offsets:
    shifted_result = softmax(logits + offset)
    print(f"offset {offset:7.1f} | maximum difference: {np.max(np.abs(reference - shifted_result)):.2e}")

## 9. Inspect the distribution, not only the winner

`argmax` returns the highest-probability token, but a distribution communicates more. The gap between the top candidates indicates whether the decision is confident or competitive. Plotting logits and probabilities side by side shows how softmax changes the scale while preserving rank.

In [ ]:
def ranked_distribution(tokens, logits):
    probabilities = softmax(logits)
    order = np.argsort(probabilities)[::-1]
    return [(tokens[i], float(logits[i]), float(probabilities[i])) for i in order]

ranking = ranked_distribution(vocabulary, logits)
for rank, (token, logit, probability) in enumerate(ranking, start=1):
    print(f"{rank}. {token:8s} logit={logit:6.3f} probability={probability:6.2%}")

### Visualize logits and probabilities

Logits can cross zero and have no fixed range. Probabilities lie between 0 and 1 and sum to 1. A token with a negative logit can still receive substantial probability because all tokens compete relative to one another.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
colors = ["#4C78A8" if value == logits.max() else "#A0CBE8" for value in logits]

axes[0].bar(vocabulary, logits, color=colors)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Unnormalized logits", ylabel="Score")
axes[1].bar(vocabulary, stable_probabilities, color=colors)
axes[1].set(title="Softmax probabilities", ylabel="Probability", ylim=(0, 1))
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 10. Changing the hidden state changes the distribution

The unembedding parameters stay fixed while the transformer's hidden state changes with context. A different context therefore produces different dot products, logits, and probabilities.

Edit the hidden state below and observe which output token becomes more likely.

In [ ]:
my_hidden_state = np.array([0.2, 1.0, -0.3, 0.7])  # Edit these features
my_logits = my_hidden_state @ unembedding + output_bias
my_probabilities = softmax(my_logits)

for token, probability in sorted(zip(vocabulary, my_probabilities), key=lambda pair: pair[1], reverse=True):
    print(f"{token:8s} {probability:6.2%}")

## 11. Sequences and batches

A transformer normally produces hidden states for every position, not only the last one. Unembedding applies the same output matrix to every hidden vector. Matrix multiplication preserves the leading axes and replaces `d_model` with `vocab_size`:

`(..., d_model) @ (d_model, vocab_size) → (..., vocab_size)`

For next-token generation we usually read the logits at the final active position. During training, every position can predict its following token in parallel.

In [ ]:
batch_size, sequence_length = 2, 3
hidden_states = rng.normal(size=(batch_size, sequence_length, d_model))
batch_logits = hidden_states @ unembedding + output_bias
batch_probabilities = softmax(batch_logits, axis=-1)

print("hidden states:", hidden_states.shape)
print("logits:       ", batch_logits.shape)
print("probabilities:", batch_probabilities.shape)
print("sums:         ", batch_probabilities.sum(axis=-1))

### Shape challenge

A model processes a batch of 8 sequences, each 32 tokens long. Its hidden size is 512 and its vocabulary has 50,000 tokens. Predict the shapes of the hidden states, unembedding matrix, logits, and probability tensor. Then use smaller arrays below to verify the general rule without allocating the large example.

In [ ]:
practice_hidden = np.zeros((2, 5, 8))
practice_unembedding = np.zeros((8, 20))
practice_logits = practice_hidden @ practice_unembedding

print("hidden:     ", practice_hidden.shape)
print("unembedding:", practice_unembedding.shape)
print("logits:     ", practice_logits.shape)

## 12. The same output layer in PyTorch

`nn.Linear(d_model, vocab_size)` packages an output weight matrix and bias. PyTorch stores weights as `(vocab_size, d_model)` and computes $hW^T+b$. This is the transpose of our NumPy storage convention.

We can copy the NumPy parameters into a layer and verify identical results.

In [ ]:
output_layer = nn.Linear(d_model, vocab_size)
with torch.no_grad():
    output_layer.weight.copy_(torch.tensor(unembedding.T, dtype=torch.float32))
    output_layer.bias.copy_(torch.tensor(output_bias, dtype=torch.float32))

hidden_torch = torch.tensor(hidden_state, dtype=torch.float32)
logits_torch = output_layer(hidden_torch)
probabilities_torch = torch.softmax(logits_torch, dim=-1)

print("PyTorch logits:       ", logits_torch)
print("PyTorch probabilities:", probabilities_torch)
print("matches NumPy:        ", np.allclose(probabilities_torch.detach().numpy(), stable_probabilities))

## 13. Probability and training loss

During training, the dataset tells us the correct next-token ID. Negative log-likelihood measures how surprised the model was by that target:

$$\text{loss} = -\log P(\text{target})$$

A confident correct prediction has probability near 1 and loss near 0. A low target probability produces a large loss. Cross-entropy applies this calculation efficiently from raw logits.

In [ ]:
target_token = "learns"
target_id = token_to_id[target_token]
target_probability = stable_probabilities[target_id]
manual_loss = -np.log(target_probability)
torch_loss = nn.functional.cross_entropy(logits_torch.unsqueeze(0), torch.tensor([target_id]))

print("target:        ", target_token)
print("probability:   ", target_probability)
print("manual loss:   ", manual_loss)
print("PyTorch loss:  ", torch_loss.item())
print("match:         ", np.allclose(manual_loss, torch_loss.item()))

## 14. One pipeline, end to end

The complete output head is only a few numerical operations. The transformer does the difficult work of constructing a context-dependent hidden state; the output head then scores every token and normalizes the scores.

In [ ]:
def next_token_distribution(hidden_state, unembedding, bias, vocabulary):
    logits = hidden_state @ unembedding + bias
    probabilities = softmax(logits)
    order = np.argsort(probabilities)[::-1]
    return {
        "logits": logits,
        "probabilities": probabilities,
        "ranking": [(vocabulary[i], float(probabilities[i])) for i in order],
    }

result = next_token_distribution(hidden_state, unembedding, output_bias, vocabulary)
print("top prediction:", result["ranking"][0])
print("distribution sums to:", result["probabilities"].sum())

## 15. What softmax does not do

Softmax does not decide which token to emit. It only produces a distribution. A decoding strategy must still select a token—perhaps by taking the maximum, sampling from the distribution, or first modifying the logits.

Softmax also does not guarantee that probabilities are well calibrated or that the highest-scoring continuation is true, safe, or sensible. Those properties depend on the model, training data, objective, and surrounding system. Notebook 06 will explore how temperature and sampling turn this distribution into generated text.

## 16. Challenges

1. **Manual logit:** Compute the `runs` logit using only elementwise multiplication and addition. Verify it against `logits`.
2. **Change a column:** Increase one unembedding column and observe which token's logit changes. Why are the other logits unchanged?
3. **Probability target:** Modify the hidden state until `writes` receives the highest probability.
4. **Stable softmax:** Explain why subtracting the mean is less safe than subtracting the maximum, even though both preserve softmax.
5. **Batching:** Extract only the final-position distribution from `batch_probabilities`. What is its shape?
6. **Loss:** Calculate the loss for every possible target token. Which target produces the smallest loss?
7. **Logit gap:** Construct two-token logits whose probabilities are approximately 90% and 10%. Hint: solve for the required logit difference using the odds ratio.

In [ ]:
# Challenge workspace: inspect the loss for every possible target.
losses = -np.log(stable_probabilities)
for token, probability, loss in sorted(zip(vocabulary, stable_probabilities, losses), key=lambda row: row[2]):
    print(f"{token:8s} probability={probability:6.2%} loss={loss:.3f}")

## Takeaways

- The final hidden state contains context-dependent features used for prediction.
- Unembedding applies one token-scoring vector per vocabulary item.
- Logits are unrestricted relative scores, not probabilities.
- Softmax produces positive values that sum to 1 while preserving rank.
- Subtracting the maximum makes softmax numerically stable without changing its output.
- Leading batch and sequence axes are preserved; the final feature axis becomes vocabulary size.
- Cross-entropy uses the target token's probability to train the hidden states and output parameters.

**Next:** *06 — Sampling and Temperature* will turn probabilities into generated tokens and control the diversity of generation.